<h1> Dataset </h1>

In [2]:
from datasets import load_dataset

dataset = load_dataset ("amalia-llm/amalia-PTradutor", "base")["train"] #https://huggingface.co/datasets/amalia-llm/amalia-PTradutor

""" Stats do Dataset """

print (dataset.features)
print (type(dataset)) #<class 'datasets.arrow_dataset.Dataset'> IMPORTANTE ser Dataset e não Columns

print (dataset[0])
print (type(dataset[0]))

{'conversations': List({'from': Value('string'), 'value': Value('string')})}
<class 'datasets.arrow_dataset.Dataset'>
{'conversations': [{'from': 'system', 'value': 'És um tradutor profissional de inglês para português. Traduz aquilo que o utilizador escrever.'}, {'from': 'user', 'value': "Brazil Collor de Mello's fatal week After three months of investigations and revelation of scandals, Brazil anxiously awaits the release of the final report of the Parliamentary Inquiry Commission which, tomorrow, should accuse President Collor de Mello of benefiting from the corruption scheme set up by Paulo César («PC») Farias. Page 10"}, {'from': 'assistant', 'value': 'Brasil A semana fatal de Collor de Mello Após três meses de investigações e revelação de escândalos, o Brasil aguarda ansiosamente a divulgação do relatório final da Comissão de Inquérito Parlamentar que, amanhã, deverá acusar o Presidente Collor de Mello de beneficiar do esquema de corrupção montado por Paulo César («PC») Farias. P

<h2> Roles do Dataset </h2>
<hr>

Esta parte é importante para percebermos se o dataset está construido de maneira uniforme com as roles tradicionais.

In [3]:
roles = set()

for exemplo in dataset["conversations"]:
    for conversas in exemplo:
        roles.add (conversas["from"])
        #print (conversas["from"])

print (roles) #{'user', 'assistant', 'system'}

{'assistant', 'system', 'user'}


Eu no meu caso, não quero <i> value </i> quero mudar para <i> content</i>. <br>
Isto é algo comum ao trabalhar com datasets.

In [4]:
def NORMALIZE (data):
    for exemplo in data["conversations"]:
        exemplo["content"] = exemplo.pop ("value") #pop método de lista

    return data

"""
map é uma função da library datasets da Hugging Face que permite aplicar transformações a um dataset 
"""
dataset = dataset.map (NORMALIZE) 

print (dataset[0])

{'conversations': [{'from': 'system', 'content': 'És um tradutor profissional de inglês para português. Traduz aquilo que o utilizador escrever.'}, {'from': 'user', 'content': "Brazil Collor de Mello's fatal week After three months of investigations and revelation of scandals, Brazil anxiously awaits the release of the final report of the Parliamentary Inquiry Commission which, tomorrow, should accuse President Collor de Mello of benefiting from the corruption scheme set up by Paulo César («PC») Farias. Page 10"}, {'from': 'assistant', 'content': 'Brasil A semana fatal de Collor de Mello Após três meses de investigações e revelação de escândalos, o Brasil aguarda ansiosamente a divulgação do relatório final da Comissão de Inquérito Parlamentar que, amanhã, deverá acusar o Presidente Collor de Mello de beneficiar do esquema de corrupção montado por Paulo César («PC») Farias. Pag. 10'}]}


<hr>

<h1> Data Pruning </h1>

Como vamos realizar Data Pruning, vamos precisar de ter os exemplos bem definidos. <br>
Nos datasets de modelos de Linguagem por norma temos sempre as chaves {"system", "user", "assistant"}. <br>
Nós para Data Pruning, não queremos isso, queremos apenas o <b> conteúdo</b> real de cada exemplo.

In [5]:
dataset_noformat = []

for exemplo in dataset["conversations"]:
    """
    Para cada exemplo vamos ficar com as 3 ou 2 chaves e juntar numa variável que depois vai ser adiconada
    à lista.
    """

    content = " ".join (x["content"] for x in exemplo) #content porque mudámos!!

    dataset_noformat.append (content.lower()) #lower para haver uma igualdade de dados

display (type(dataset_noformat))
print (dataset_noformat[4054])
display (dataset_noformat[:5])

list

és um tradutor especializado em inglês-português. a tua tarefa é traduzir o texto do utilizador para português. q - are the neologisms you use in your poetry all your creation, or do you also collect them from the mouths of the people, here in the pantanal? m. b. -- ah yes, i discover many terms in people's speech. what i call plural syntax is practiced by illiterate men, who express themselves in a different way than we do. this syntax of the people, quite plural -- because they are not guided by grammar and therefore can speak in different ways -- i really like that. for me it's the same thing as discovering camilo's syntax, one is erudite, constructed, and the other primary, but each has its own beauty. another thing i like is listening to the children's conversations. i have a poem in my new book where i try to explain children's poetry: «i heard the color of the birds». children practice these types of phrases all the time. a child's language is always a birth. i exploited my five

["és um tradutor profissional de inglês para português. traduz aquilo que o utilizador escrever. brazil collor de mello's fatal week after three months of investigations and revelation of scandals, brazil anxiously awaits the release of the final report of the parliamentary inquiry commission which, tomorrow, should accuse president collor de mello of benefiting from the corruption scheme set up by paulo césar («pc») farias. page 10 brasil a semana fatal de collor de mello após três meses de investigações e revelação de escândalos, o brasil aguarda ansiosamente a divulgação do relatório final da comissão de inquérito parlamentar que, amanhã, deverá acusar o presidente collor de mello de beneficiar do esquema de corrupção montado por paulo césar («pc») farias. pag. 10",
 'por favor, traduz este texto de inglês para português:\nthe state of innocence, symbolized yesterday in the brussels demonstration by the white of the clothes, often does not survive the violence and hypocrisy of the w

<h2> Duplicados usando Hashes </h2>
<hr> 

https://en.wikipedia.org/wiki/Hash_function <br>
Um Hash é uma representação de algo a partir de uma assinatura única. <br>
É perfeito para duplicados porque cada exemplo vai ter uma assinatura única mas se houver exemplos iguais, ambos vão ter assinaturas iguais. <br>
É uma maneira rápida e barata de remover duplicados. <br>
Não compensa comparar exemplo a exemplo porque é muito mais caro!

In [5]:
import hashlib #https://docs.python.org/3/library/hashlib.html

"""
Uma maneira muito boa de ver os duplicados é usar set.
Set é uma maneira de armazenar dados em Python que não permite duplicados.
Dito isto, se houver alguma assinatura duplicada, não irá ser adicionada ao set.
"""
DUPLICADOS = set()
HASH = [] #Para vermos realmente o que o algoritmo Hash faz

for exemplo in dataset_noformat:

    hs = hashlib.blake2b (exemplo.encode()).hexdigest()

    HASH.append (hs)
    DUPLICADOS.add (hs)


print (len(HASH)) #Aqui tem que obrigatoriamente que dar o número de exemplos do dataset

"""
Se len(duplicados) == len(dataset) é porque NÃO há exemplos duplicados.
Caso contrário, há duplicados.
"""
print (len(DUPLICADOS))
print (f"Número de Duplicados: {len(HASH) - len(DUPLICADOS)}")

50000
50000
Número de Duplicados: 0


<h2> Duplicados Lexicais </h2>
<hr>

https://www.pinecone.io/learn/series/faiss/locality-sensitive-hashing/ <br>
Os Duplicados Lexicais representam <i> near-duplicates</i> a nivel Lexical. <br>
Desta maneira conseguimos perceber se um exemplo é parecido a outro a nivel Lexical. <br>

Para esta etapa usamos uma técnica parecida ao Hash.. MinHash. <br>
<b>MinHash</b> aplica um Hash a cada palavra ou <b> shingle</b>. <br>
A ideia é depois comparar todos esses Hashes exemplo a exemplo mas isto é muito caro. <br>

Por isso usamos <b>Locality Sensitive Hashing</b> que utiliza partes dessas assinaturas para colocar exemplos parecidos num grupo e assim só compara todas as assinaturas com esses exemplos.


In [6]:
from datasketch import MinHash
##### Exemplo para Perceber

MinHash_1 = MinHash ()
MinHash_2 = MinHash ()

#Exemplo 1
for palavra in dataset_noformat[0].split ():
    MinHash_1.update (palavra.encode ("utf-8"))

#Exemplo 2
for palavra in dataset_noformat[1].split ():
    MinHash_2.update (palavra.encode ("utf-8"))


print (f"Similaridade de Jaccard entre o exemplo 0 e o exemplo 1 do dataset: {MinHash_1.jaccard (MinHash_2) * 100}%")


"""
Agora imaginemos isto a uma escala de centenas de milhares de exemplos!!
É aqui entra Locality Sensitive Hashing.
"""

Similaridade de Jaccard entre o exemplo 0 e o exemplo 1 do dataset: 4.6875%


'\nAgora imaginemos isto a uma escala de centenas de milhares de exemplos!!\nÉ aqui entra Locality Sensitive Hashing.\n'

In [7]:
"""
Para similaridade lexical, usar apenas palavras não acaba por ser a melhor solução.
Estamos apenas a comparar palavra a palavra sem ter todo o contexto por trás que acaba
por ser relevante para a compreensão lexical.

Por esse motivo usamos Shingles, que são nada mais nada menos do que n-grams.
    Exemplo:

        {"o", "gato", "comeu", "o", "rato"}   |    {"o", "gato", "não", "comeu" "o" "rato"}    
    
Se comparámos palavra a palavra claramente ambas as frases são parecidas mas qualquer humano consegue
perceber que não tem o mesmo valor lexical.
"""

def SHINGLES (texto, k = 5):

    palavras = texto.split()

    return [" ".join(palavras [i : i + k]) for i in range (len (palavras) - k + 1)]

x = SHINGLES (dataset_noformat[0])
print (x)
print (len(x))

['és um tradutor profissional de', 'um tradutor profissional de inglês', 'tradutor profissional de inglês para', 'profissional de inglês para português.', 'de inglês para português. traduz', 'inglês para português. traduz aquilo', 'para português. traduz aquilo que', 'português. traduz aquilo que o', 'traduz aquilo que o utilizador', 'aquilo que o utilizador escrever.', 'que o utilizador escrever. brazil', 'o utilizador escrever. brazil collor', 'utilizador escrever. brazil collor de', "escrever. brazil collor de mello's", "brazil collor de mello's fatal", "collor de mello's fatal week", "de mello's fatal week after", "mello's fatal week after three", 'fatal week after three months', 'week after three months of', 'after three months of investigations', 'three months of investigations and', 'months of investigations and revelation', 'of investigations and revelation of', 'investigations and revelation of scandals,', 'and revelation of scandals, brazil', 'revelation of scandals, brazil a

In [9]:
from datasketch import MinHashLSH #https://github.com/ekzhu/datasketch

"""
Aqui usamos LSH para o mesmo meter em grupos os exemplos candidatos a serem parecidos.
Ainda não é Similaridade Jaccard!!
"""

MinimunHashLSH = {}

LocalitySensitiveHashing = MinHashLSH (threshold = 0.55)

for idx, exemplo in enumerate (dataset_noformat):

    MinimunHash = MinHash () #Para cada exemplo criamos um MinHash novo

    for ngrams in SHINGLES (exemplo):

        MinimunHash.update (ngrams.encode ("utf-8")) #MinHash de cada Shingle

    MinimunHashLSH [f"Exemplo {idx}"] = MinimunHash #Guardamos num dict
    LocalitySensitiveHashing.insert (f"Exemplo {idx}", MinimunHash) #Guardamos no idx do LSH

### 3min
print (MinimunHashLSH["Exemplo 223"])

In [10]:
"""
Agora com o dict e com o indice LSH podemos percorrer cada exemplo e encontrar os candidatos a mais semelhantes.
É preciso ter atenção que isto ainda não são os mais semelhantes, são os CANDIDATOS a mais semelhantes.
"""

Candidatos = []

for nome, minash in MinimunHashLSH.items():

    #print (f"{nome} -> {minash}")

    LSH = LocalitySensitiveHashing.query (minash)

    Candidatos.append (LSH)


"""
LSH devolve uma lista com o exemplo que pedimos para procurar os candidatos mais parecidos e os 
candidatos mais parecidos caso exista. 
Nesse sentido podemos procurar os candidatos a dups procurando pela len.
"""

SimilJaccard_Set = set()

for x in Candidatos:

    dups = len(x)

    if dups >= 2:

        MinimunHash_1 = MinimunHashLSH[x[0]]
        MinimunHash_2 = MinimunHashLSH[x[1]]

        SimilJaccard = MinimunHash_1.jaccard (MinimunHash_2)

        SimilJaccard_Set.add (f"{x}: {SimilJaccard * 100}%")

print (SimilJaccard_Set) #Se reparámos há exemplos que têm similaridade relativamente baixa, o que comprova que LSH fornece apenas os candidatos e não os mais semelhantes de todo

{"['Exemplo 47987', 'Exemplo 9498']: 85.15625%", "['Exemplo 27209', 'Exemplo 24985']: 29.6875%", "['Exemplo 32697', 'Exemplo 34977']: 67.96875%", "['Exemplo 22884', 'Exemplo 8769']: 75.0%", "['Exemplo 46269', 'Exemplo 9025']: 53.125%", "['Exemplo 8052', 'Exemplo 42065']: 43.75%", "['Exemplo 8892', 'Exemplo 34578']: 73.4375%", "['Exemplo 47817', 'Exemplo 6623']: 64.84375%", "['Exemplo 36517', 'Exemplo 16852']: 57.03125%", "['Exemplo 43021', 'Exemplo 3250']: 76.5625%", "['Exemplo 755', 'Exemplo 216']: 72.65625%", "['Exemplo 48999', 'Exemplo 7825']: 75.0%", "['Exemplo 23342', 'Exemplo 48663']: 52.34375%", "['Exemplo 6758', 'Exemplo 43427']: 56.25%", "['Exemplo 26137', 'Exemplo 35789']: 31.25%", "['Exemplo 4737', 'Exemplo 20656']: 64.84375%", "['Exemplo 3435', 'Exemplo 8371']: 67.96875%", "['Exemplo 9816', 'Exemplo 28047']: 78.125%"}


<hr>
<hr>

<h1> Heurísticas </h1>

<h3> Length Profiler </h3>

In [8]:
heu = {}

for idx, exemplo in enumerate (dataset_noformat):

    WORDS = exemplo.split()
    WORDS_UNIQUE = set (WORDS)

    TTR = len (WORDS_UNIQUE) / len (WORDS) * 100
    
    """ 
    De seguida vamos calcular o Type Token Ratio = Número de Palavras Únicas / Número de Palavras.
    Porém, isto calculado de maneira bruta pode induzir em erro porque exemplos maiores podem facilmente ter um TTR baixo mas não quer dizer que sejam maus exemplos. 
    Solução -> Calcular o TTR de acordo com o comprimento específico
    """

    TTR_WORDS = WORDS [:100]
    TTR_WORDS_UNIQUE = set (TTR_WORDS)
    TTR_50 = len (TTR_WORDS_UNIQUE) / len (TTR_WORDS) * 100

    heu[f"Exemplo {idx}"] = {
        
        "Número de Words do Exemplo": len (WORDS), # Sem ter o espaço
        "Número de Words Únicas do Exemplo": len (WORDS_UNIQUE),

        "TTR": TTR, 
        "TTR @ 50": TTR_50

        }

print (heu["Exemplo 34444"])
print (heu["Exemplo 322"])
print (heu["Exemplo 1093"])

print (heu.items())

{'Número de Words do Exemplo': 369, 'Número de Words Únicas do Exemplo': 238, 'TTR': 64.49864498644986, 'TTR @ 50': 69.0}
{'Número de Words do Exemplo': 493, 'Número de Words Únicas do Exemplo': 302, 'TTR': 61.257606490872206, 'TTR @ 50': 86.0}
{'Número de Words do Exemplo': 473, 'Número de Words Únicas do Exemplo': 301, 'TTR': 63.63636363636363, 'TTR @ 50': 77.0}
dict_items([('Exemplo 0', {'Número de Words do Exemplo': 120, 'Número de Words Únicas do Exemplo': 85, 'TTR': 70.83333333333334, 'TTR @ 50': 77.0}), ('Exemplo 1', {'Número de Words do Exemplo': 483, 'Número de Words Únicas do Exemplo': 317, 'TTR': 65.63146997929607, 'TTR @ 50': 75.0}), ('Exemplo 2', {'Número de Words do Exemplo': 593, 'Número de Words Únicas do Exemplo': 348, 'TTR': 58.68465430016864, 'TTR @ 50': 78.0}), ('Exemplo 3', {'Número de Words do Exemplo': 498, 'Número de Words Únicas do Exemplo': 316, 'TTR': 63.45381526104418, 'TTR @ 50': 81.0}), ('Exemplo 4', {'Número de Words do Exemplo': 182, 'Número de Words Úni

In [9]:
import numpy as np

TTR = []
TTR_50 = []

for x in heu:

    TTR.append (heu[x]["TTR"])
    TTR_50.append (heu[x]["TTR @ 50"])
    

TTR_P1 = np.percentile (TTR, 1)
TTR_P5 = np.percentile (TTR, 5)
TTR_P25 = np.percentile (TTR, 25)
TTR_P50 = np.percentile (TTR, 50)
TTR_P75 = np.percentile (TTR, 75)
TTR_P90 = np.percentile (TTR, 90)
TTR_P95 = np.percentile (TTR, 95)
TTR_P99 = np.percentile (TTR, 99)

print ("Percentis TTR:")
print (f"1% dos Exemplos têm menos de {TTR_P1} TTR.") 
print (f"5% dos Exemplos têm menos de {TTR_P5} TTR.") 
print (f"25% dos Exemplos têm menos de {TTR_P25} TTR.") 
print (f"50% dos Exemplos têm menos de {TTR_P50} TTR.") 
print (f"75% dos Exemplos têm menos de {TTR_P75} TTR.") 
print (f"90% dos Exemplos têm menos de {TTR_P90} TTR.") 
print (f"95% dos Exemplos têm menos de {TTR_P95} TTR.") 
print (f"99% dos Exemplos têm menos de {TTR_P99} TTR.") 

for idx, values in heu.items():
    
    TTR = values["TTR"]

    if TTR <= TTR_P1:

        values["Grupo Percentil TTR"] = "<P1"

    elif TTR_P1 < TTR <= TTR_P5:

        values["Grupo Percentil TTR"] = "P1 - P5"

    elif TTR_P5 < TTR <= TTR_P25:

        values["Grupo Percentil TTR"] = "P5 - P25"

    elif TTR_P25 < TTR <= TTR_P50:

        values["Grupo Percentil TTR"] = "P25 - P50"

    elif TTR_P50 < TTR <= TTR_P75:

        values["Grupo Percentil TTR"] = "P50 - P75"

    elif TTR_P75 < TTR <= TTR_P90:

        values["Grupo Percentil TTR"] = "P75 - P90"

    elif TTR_P90 < TTR <= TTR_P95:

        values["Grupo Percentil TTR"] = "P90 - P95"

    elif TTR_P95 < TTR <= TTR_P99:

        values["Grupo Percentil TTR"] = "P95 - P99"

    else:

        values["Grupo Percentil TTR"] = ">P99"


"""
#####################################################################################################################################################################
#####################################################################################################################################################################
#####################################################################################################################################################################
#####################################################################################################################################################################
#####################################################################################################################################################################
#####################################################################################################################################################################
"""

TTR50_P1 = np.percentile (TTR_50, 1)
TTR50_P5 = np.percentile (TTR_50, 5)
TTR50_P25 = np.percentile (TTR_50, 25)
TTR50_P50 = np.percentile (TTR_50, 50)
TTR50_P75 = np.percentile (TTR_50, 75)
TTR50_P90 = np.percentile (TTR_50, 90)
TTR50_P95 = np.percentile (TTR_50, 95)
TTR50_P99 = np.percentile (TTR_50, 99)

print ("\nPercentis TTR @ 50:")
print (f"1% dos Exemplos têm menos de {TTR50_P1} TTR @ 50.") 
print (f"5% dos Exemplos têm menos de {TTR50_P5} TTR @ 50.") 
print (f"25% dos Exemplos têm menos de {TTR50_P25} TTR @ 50.") 
print (f"50% dos Exemplos têm menos de {TTR50_P50} TTR @ 50.") 
print (f"75% dos Exemplos têm menos de {TTR50_P75} TTR @ 50.") 
print (f"90% dos Exemplos têm menos de {TTR50_P90} TTR @ 50.") 
print (f"95% dos Exemplos têm menos de {TTR50_P95} TTR @ 50.") 
print (f"99% dos Exemplos têm menos de {TTR50_P99} TTR @ 50.") 

for idx, values in heu.items ():

    TTR_50 = values["TTR @ 50"]

    if TTR_50 <= TTR50_P1:

        values["Grupo Percentil TTR @ 50"] = "<P1"

    elif TTR50_P1 < TTR_50 <= TTR50_P5:

        values["Grupo Percentil TTR @ 50"] = "P1 - P5"

    elif TTR50_P5 < TTR_50 <= TTR50_P25:

        values["Grupo Percentil TTR @ 50"] = "P5 - P25"

    elif TTR50_P25 < TTR_50 <= TTR50_P50:

        values["Grupo Percentil TTR @ 50"] = "P25 - P50"

    elif TTR50_P50 < TTR_50 <= TTR50_P75:

        values["Grupo Percentil TTR @ 50"] = "P50 - P75"

    elif TTR50_P75 < TTR_50 <= TTR50_P90:

        values["Grupo Percentil TTR @ 50"] = "P75 - P90"

    elif TTR50_P90 < TTR_50 <= TTR50_P95:

        values["Grupo Percentil TTR @ 50"] = "P90 - P95"

    elif TTR50_P95 < TTR_50 <= TTR50_P99:

        values["Grupo Percentil TTR @ 50"] = "P95 - P99"

    else:

        values["Grupo Percentil TTR @ 50"] = ">P99"


display (heu)

Percentis TTR:
1% dos Exemplos têm menos de 52.04911863211704 TTR.
5% dos Exemplos têm menos de 56.48535564853556 TTR.
25% dos Exemplos têm menos de 61.2691466083151 TTR.
50% dos Exemplos têm menos de 64.94845360824742 TTR.
75% dos Exemplos têm menos de 71.7948717948718 TTR.
90% dos Exemplos têm menos de 78.62595419847328 TTR.
95% dos Exemplos têm menos de 82.02247191011236 TTR.
99% dos Exemplos têm menos de 87.12871287128714 TTR.

Percentis TTR @ 50:
1% dos Exemplos têm menos de 65.0 TTR @ 50.
5% dos Exemplos têm menos de 70.0 TTR @ 50.
25% dos Exemplos têm menos de 74.48979591836735 TTR @ 50.
50% dos Exemplos têm menos de 78.0 TTR @ 50.
75% dos Exemplos têm menos de 81.0 TTR @ 50.
90% dos Exemplos têm menos de 83.0 TTR @ 50.
95% dos Exemplos têm menos de 85.0 TTR @ 50.
99% dos Exemplos têm menos de 88.0 TTR @ 50.


{'Exemplo 0': {'Número de Words do Exemplo': 120,
  'Número de Words Únicas do Exemplo': 85,
  'TTR': 70.83333333333334,
  'TTR @ 50': 77.0,
  'Grupo Percentil TTR': 'P50 - P75',
  'Grupo Percentil TTR @ 50': 'P25 - P50'},
 'Exemplo 1': {'Número de Words do Exemplo': 483,
  'Número de Words Únicas do Exemplo': 317,
  'TTR': 65.63146997929607,
  'TTR @ 50': 75.0,
  'Grupo Percentil TTR': 'P50 - P75',
  'Grupo Percentil TTR @ 50': 'P25 - P50'},
 'Exemplo 2': {'Número de Words do Exemplo': 593,
  'Número de Words Únicas do Exemplo': 348,
  'TTR': 58.68465430016864,
  'TTR @ 50': 78.0,
  'Grupo Percentil TTR': 'P5 - P25',
  'Grupo Percentil TTR @ 50': 'P25 - P50'},
 'Exemplo 3': {'Número de Words do Exemplo': 498,
  'Número de Words Únicas do Exemplo': 316,
  'TTR': 63.45381526104418,
  'TTR @ 50': 81.0,
  'Grupo Percentil TTR': 'P25 - P50',
  'Grupo Percentil TTR @ 50': 'P50 - P75'},
 'Exemplo 4': {'Número de Words do Exemplo': 182,
  'Número de Words Únicas do Exemplo': 125,
  'TTR': 68.

<h3> Repetition Profiler </h3>

In [10]:
#### N Grams Repetidos

def NGRAMS (texto, k):

    palavras = texto.split()

    return [" ".join(palavras [i : i + k]) for i in range (len (palavras) - k + 1)]

for idx, value in heu.items():

    exemplo = dataset_noformat [int(idx.split()[-1])] #Genialidade 

    """
    Bi Gram
    """

    n = NGRAMS (exemplo, k = 2)

    ngrams = []
    nodups = set()

    for x in n:

        ngrams.append (x)
        nodups.add (x)

    value["Número de Bi-Grams do Exemplo"] = len(ngrams)
    value["Número de Bi-Grams Repetidos do Exemplo"] = len(ngrams) - len(nodups)
    value["% de Bi-Grams Repetidos do Exemplo"] = (1 - (len(nodups) / len(ngrams))) * 100

    """
    Tri Gram
    """

    n = NGRAMS (exemplo, k = 3)

    ngrams = []
    nodups = set()

    for x in n:

        ngrams.append (x)
        nodups.add (x)
        
    value["Número de Tri-Grams do Exemplo"] = len(ngrams)
    value["Número de Tri-Grams Repetidos do Exemplo"] = len(ngrams) - len(nodups)
    value["% de Tri-Grams Repetidos do Exemplo"] = (1 - (len(nodups) / len(ngrams))) * 100


display (heu["Exemplo 29334"])

{'Número de Words do Exemplo': 121,
 'Número de Words Únicas do Exemplo': 90,
 'TTR': 74.3801652892562,
 'TTR @ 50': 76.0,
 'Grupo Percentil TTR': 'P75 - P90',
 'Grupo Percentil TTR @ 50': 'P25 - P50',
 'Número de Bi-Grams do Exemplo': 120,
 'Número de Bi-Grams Repetidos do Exemplo': 6,
 '% de Bi-Grams Repetidos do Exemplo': 5.000000000000004,
 'Número de Tri-Grams do Exemplo': 119,
 'Número de Tri-Grams Repetidos do Exemplo': 2,
 '% de Tri-Grams Repetidos do Exemplo': 1.6806722689075682}

In [11]:

bigram_ratio = []
trigram_ratio = []

for idx, value in heu.items():


    bigram_ratio.append (value["% de Bi-Grams Repetidos do Exemplo"])
    trigram_ratio.append (value["% de Tri-Grams Repetidos do Exemplo"])


#### Bi Gram Descriminação
bigram_ratio_P1 = np.percentile (bigram_ratio, 1)
bigram_ratio_P5 = np.percentile (bigram_ratio, 5)
bigram_ratio_P25 = np.percentile (bigram_ratio, 25)
bigram_ratio_P50 = np.percentile (bigram_ratio, 50)
bigram_ratio_P75 = np.percentile (bigram_ratio, 75)
bigram_ratio_P90 = np.percentile (bigram_ratio, 90)
bigram_ratio_P95 = np.percentile (bigram_ratio, 95)
bigram_ratio_P99 = np.percentile (bigram_ratio, 99)
print (f"Bi Gram Stats: \nP1: {bigram_ratio_P1}\nP5: {bigram_ratio_P5}\nP25: {bigram_ratio_P25}\nP50: {bigram_ratio_P50}\nP75: {bigram_ratio_P75}\nP90: {bigram_ratio_P90}\nP95: {bigram_ratio_P95}\nP99: {bigram_ratio_P99}")

#### Tri Gram Descriminação
trigram_ratio_P1 = np.percentile (trigram_ratio, 1)
trigram_ratio_P5 = np.percentile (trigram_ratio, 5)
trigram_ratio_P25 = np.percentile (trigram_ratio, 25)
trigram_ratio_P50 = np.percentile (trigram_ratio, 50)
trigram_ratio_P75 = np.percentile (trigram_ratio, 75)
trigram_ratio_P90 = np.percentile (trigram_ratio, 90)
trigram_ratio_P95 = np.percentile (trigram_ratio, 95)
trigram_ratio_P99 = np.percentile (trigram_ratio, 99)
print (f"\nTri Gram Stats: \nP1: {trigram_ratio_P1}\nP5: {trigram_ratio_P5}\nP25: {trigram_ratio_P25}\nP50 {trigram_ratio_P50}\nP75: {trigram_ratio_P75}\nP90: {trigram_ratio_P90}\nP95: {trigram_ratio_P95}\nP99: {trigram_ratio_P99}")

"""
############################################################################################################################################################################################
############################################################################################################################################################################################
############################################################################################################################################################################################
############################################################################################################################################################################################
############################################################################################################################################################################################
############################################################################################################################################################################################
"""

for idx, values in heu.items():
    
#######################################################
                    #BI-GRAMS
#######################################################

    bigram_ratio = values["% de Bi-Grams Repetidos do Exemplo"]

    if bigram_ratio <= bigram_ratio_P1:

        values["Grupo Percentil % de Bi-Grams Repetidos do Exemplo"] = "<P1"

    elif bigram_ratio_P1 < bigram_ratio <= bigram_ratio_P5:

        values["Grupo Percentil % de Bi-Grams Repetidos do Exemplo"] = "P1 - P5"

    elif bigram_ratio_P5 < bigram_ratio <= bigram_ratio_P25:

        values["Grupo Percentil % de Bi-Grams Repetidos do Exemplo"] = "P5 - P25"

    elif bigram_ratio_P25 < bigram_ratio <= bigram_ratio_P50:

        values["Grupo Percentil % de Bi-Grams Repetidos do Exemplo"] = "P25 - P50"

    elif bigram_ratio_P50 < bigram_ratio <= bigram_ratio_P75:

        values["Grupo Percentil % de Bi-Grams Repetidos do Exemplo"] = "P50 - P75"

    elif bigram_ratio_P75 < bigram_ratio <= bigram_ratio_P90:

        values["Grupo Percentil % de Bi-Grams Repetidos do Exemplo"] = "P75 - P90"

    elif bigram_ratio_P90 < bigram_ratio <= bigram_ratio_P95:

        values["Grupo Percentil % de Bi-Grams Repetidos do Exemplo"] = "P90 - P95"

    elif bigram_ratio_P95 < bigram_ratio <= bigram_ratio_P99:

        values["Grupo Percentil % de Bi-Grams Repetidos do Exemplo"] = "P95 - P99"

    else:

        values["Grupo Percentil % de Bi-Grams Repetidos do Exemplo"] = ">P99"

#######################################################
                    #TRI-GRAMS
#######################################################

    trigram_ratio = values["% de Tri-Grams Repetidos do Exemplo"]

    if trigram_ratio <= trigram_ratio_P1:

        values["Grupo Percentil % de Tri-Grams Repetidos do Exemplo"] = "<P1"

    elif trigram_ratio_P1 < trigram_ratio <= trigram_ratio_P5:

        values["Grupo Percentil % de Tri-Grams Repetidos do Exemplo"] = "P1 - P5"

    elif trigram_ratio_P5 < trigram_ratio <= trigram_ratio_P25:

        values["Grupo Percentil % de Tri-Grams Repetidos do Exemplo"] = "P5 - P25"

    elif trigram_ratio_P25 < trigram_ratio <= trigram_ratio_P50:

        values["Grupo Percentil % de Tri-Grams Repetidos do Exemplo"] = "P25 - P50"

    elif trigram_ratio_P50 < trigram_ratio <= trigram_ratio_P75:

        values["Grupo Percentil % de Tri-Grams Repetidos do Exemplo"] = "P50 - P75"

    elif trigram_ratio_P75 < trigram_ratio <= trigram_ratio_P90:

        values["Grupo Percentil % de Tri-Grams Repetidos do Exemplo"] = "P75 - P90"

    elif trigram_ratio_P90 < trigram_ratio <= trigram_ratio_P95:

        values["Grupo Percentil % de Tri-Grams Repetidos do Exemplo"] = "P90 - P95"

    elif trigram_ratio_P95 < trigram_ratio <= trigram_ratio_P99:

        values["Grupo Percentil % de Tri-Grams Repetidos do Exemplo"] = "P95 - P99"

    else:

        values["Grupo Percentil % de Tri-Grams Repetidos do Exemplo"] = ">P99"

display (heu["Exemplo 5454"])

Bi Gram Stats: 
P1: 0.0
P5: 0.5680213148433542
P25: 2.7272727272727226
P50: 4.402515723270439
P75: 6.33187772925764
P90: 8.530899721658015
P95: 10.309278350515461
P99: 15.277777777777779

Tri Gram Stats: 
P1: 0.0
P5: 0.0
P25: 0.0
P50 0.6993006993006978
P75: 1.6326530612244872
P90: 2.985074626865669
P95: 4.262911864733502
P99: 8.25688073394495


{'Número de Words do Exemplo': 266,
 'Número de Words Únicas do Exemplo': 184,
 'TTR': 69.17293233082707,
 'TTR @ 50': 81.0,
 'Grupo Percentil TTR': 'P50 - P75',
 'Grupo Percentil TTR @ 50': 'P50 - P75',
 'Número de Bi-Grams do Exemplo': 265,
 'Número de Bi-Grams Repetidos do Exemplo': 10,
 '% de Bi-Grams Repetidos do Exemplo': 3.7735849056603765,
 'Número de Tri-Grams do Exemplo': 264,
 'Número de Tri-Grams Repetidos do Exemplo': 0,
 '% de Tri-Grams Repetidos do Exemplo': 0.0,
 'Grupo Percentil % de Bi-Grams Repetidos do Exemplo': 'P25 - P50',
 'Grupo Percentil % de Tri-Grams Repetidos do Exemplo': '<P1'}

In [32]:
def SCORES_RIQUEZA_LEX (percentil):

    scores = {
        "<P1": 0,
        "P1 - P5": 1,
        "P5 - P25": 2,
        "P25 - P50": 3,
        "P50 - P75": 4.5,
        "P75 - P90": 4,
        "P90 - P95": 4,
        "P95 - P99": 5,
        ">P99": 5
    }

    return scores [percentil] 


for idx, valores in heu.items():

    """
    Riqueza Lexical Eval
    """

    gp_TTR = valores ["Grupo Percentil TTR"]
    gp_TTR_50 = valores ["Grupo Percentil TTR @ 50"]

    score_TTR = SCORES_RIQUEZA_LEX (gp_TTR)
    score_TTR_50 = SCORES_RIQUEZA_LEX (gp_TTR_50)

    riqueza_lex_score = (score_TTR + score_TTR_50) / 2

    valores ["Score TTR do Exemplo"] = score_TTR
    valores ["Score TTR @ 50 do Exemplo"] = score_TTR_50
    valores ["Score Riqueza Lexical do Exemplo"] = riqueza_lex_score

"""
####################################################################################################################################################
####################################################################################################################################################
####################################################################################################################################################
####################################################################################################################################################
"""

def SCORES_REPETITION (percentil):

    scores = {
        "<P1": 5,
        "P1 - P5": 5,
        "P5 - P25": 4,
        "P25 - P50": 4,
        "P50 - P75": 3.5,
        "P75 - P90": 3,
        "P90 - P95": 2,
        "P95 - P99": 1,
        ">P99": 0
    }

    return scores [percentil] 


for idx, valores in heu.items():

    """
    Repetição Eval
    """

    gp_Bigram = valores ["Grupo Percentil % de Bi-Grams Repetidos do Exemplo"]
    gp_Trigram = valores ["Grupo Percentil % de Tri-Grams Repetidos do Exemplo"]

    score_Bigram = SCORES_REPETITION (gp_Bigram)
    score_Trigram = SCORES_REPETITION (gp_Trigram)

    rep_score = (score_Bigram + score_Trigram) / 2

    valores ["Score Repetição Bigram do Exemplo"] = score_Bigram
    valores ["Score Repetição Trigram do Exemplo"] = score_Trigram
    valores ["Score Repetição do Exemplo"] = rep_score


    ### Score Final

    valores ["Heuristic Score"] = (valores ["Score Riqueza Lexical do Exemplo"] + valores ["Score Repetição do Exemplo"]) / 2

In [46]:
display (heu ["Exemplo 4545"])
print (dataset_noformat[4545])
print (dataset["conversations"][4545])

{'Número de Words do Exemplo': 189,
 'Número de Words Únicas do Exemplo': 124,
 'TTR': 65.60846560846561,
 'TTR @ 50': 76.0,
 'Grupo Percentil TTR': 'P50 - P75',
 'Grupo Percentil TTR @ 50': 'P25 - P50',
 'Número de Bi-Grams do Exemplo': 188,
 'Número de Bi-Grams Repetidos do Exemplo': 18,
 '% de Bi-Grams Repetidos do Exemplo': 9.57446808510638,
 'Número de Tri-Grams do Exemplo': 187,
 'Número de Tri-Grams Repetidos do Exemplo': 6,
 '% de Tri-Grams Repetidos do Exemplo': 3.208556149732622,
 'Grupo Percentil % de Bi-Grams Repetidos do Exemplo': 'P90 - P95',
 'Grupo Percentil % de Tri-Grams Repetidos do Exemplo': 'P90 - P95',
 'Score TTR do Exemplo': 4.5,
 'Score TTR @ 50 do Exemplo': 3,
 'Score Riqueza Lexical do Exemplo': 3.75,
 'Score Repetição Bigram do Exemplo': 2,
 'Score Repetição Trigram do Exemplo': 2,
 'Score Repetição do Exemplo': 2.0,
 'Heuristic Score': 2.875}

és um tradutor especializado em inglês-português. a tua tarefa é traduzir o texto do utilizador para português. the three proposals on party financing what can change helena pereira - donation limits the three proposals from the executive, psd and pcp are in agreement that the amount of pecuniary donations granted by citizens does not exceed the amount of 30 minimum wages national. however, socialists allow contributions from legal entities of up to 100 minimum wages. in the case of an electoral campaign, donations from individuals may amount to 100 minimum wages, in accordance with the government and psd proposals. as três propostas sobre o financiamento de partidos o que pode mudar helena pereira -limites de donativos as três propostas do executivo, psd e pcp estão de acordo em que o montante de donativos de natureza pecuniária concedidos por cidadãos não exceda o montante de 30 salários mínimos nacionais. no entanto, os socialistas admitem contribuições, por parte de pessoas colecti

In [51]:
remove = []

for idx, valores in heu.items ():

    if valores ["Score Riqueza Lexical do Exemplo"] <= 2 and valores ["Score Repetição do Exemplo"] <= 2:

        remove.append (idx)


print (len(remove))
print (remove [:10]) #['Exemplo 35', 'Exemplo 78', 'Exemplo 109', 'Exemplo 156', 'Exemplo 179', 'Exemplo 236', 'Exemplo 265', 'Exemplo 303', 'Exemplo 319', 'Exemplo 373']
print (dataset_noformat[35])


2041
['Exemplo 35', 'Exemplo 78', 'Exemplo 109', 'Exemplo 156', 'Exemplo 179', 'Exemplo 236', 'Exemplo 265', 'Exemplo 303', 'Exemplo 319', 'Exemplo 373']
traduz isto de inglês para português:
at one point, however, they all tell the judge the same thing. jean-pierre bernès, former sporting director of the om, stated on 17 june 1994 and 17 february 1995 that he had no power to refuse to sign an invoice, that the payment mechanisms [amounts of money embezzled] were known to tapie, laroche and fellous and that he limited himself to signing the invoices with the agreement of tapie and laroche and at their request. jean-louis levreau, former vice-president of the club, stated, for his part, on april 27, 1995, that those responsible for decisions at om were, in this order, bernard tapie, jean-pierre bernès, alain laroche and elie fellous. when he went to bernard tapie (to confirm the payment of sums exceeding one hundred thousand francs), he told him to trust bernès and laroche. elie fellous

<h1> Embeddings </h1>

In [6]:
from transformers import AutoModel

MODELO = AutoModel.from_pretrained ("jinaai/jina-embeddings-v5-text-small", trust_remote_code = True)



Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 480.10it/s]


In [14]:
text_embeddings = MODELO.encode (texts = dataset_noformat, task = "text-matching")

RuntimeError: [enforce fail at alloc_cpu.cpp:117] data. DefaultCPUAllocator: not enough memory: you tried to allocate 106188800000 bytes.

In [11]:
import torch.nn.functional as F


print (text_embeddings)
print (text_embeddings.shape)

print (text_embeddings[0])
print (text_embeddings[0].shape)

cos_simil = F.cosine_similarity (text_embeddings[3], text_embeddings[4], dim = 0)

print (cos_simil)

tensor([[-0.0024,  0.0457, -0.0464,  ..., -0.0304,  0.0286,  0.0386],
        [ 0.0017,  0.0427, -0.0540,  ..., -0.0376,  0.0182,  0.0552],
        [-0.0417,  0.0077, -0.0447,  ..., -0.0557,  0.0223,  0.0469],
        [-0.0947,  0.0967, -0.0454,  ..., -0.0391,  0.0082,  0.0608],
        [-0.0415,  0.0532, -0.0422,  ..., -0.0337,  0.0525,  0.0645]],
       dtype=torch.bfloat16)
torch.Size([5, 1024])
tensor([-0.0024,  0.0457, -0.0464,  ..., -0.0304,  0.0286,  0.0386],
       dtype=torch.bfloat16)
torch.Size([1024])
tensor(0.5547, dtype=torch.bfloat16)


In [13]:
import faiss
import numpy as np


emb = text_embeddings.float().cpu().numpy()

faiss.normalize_L2 (emb)

index = faiss.IndexFlatIP (emb.shape[1])
index.add (emb)

